# Void Clipex bot (free Colab runner)

1. Left side key icon (Secrets) -> Add new secret -> Name: `TELEGRAM_BOT_TOKEN` -> Value: bot token -> Notebook access ON.
2. Runtime -> **Run all**.
3. Wait 5-8 minutes, then message your bot on Telegram.
4. Keep the tab open. Free Colab allows roughly 12 hours; after that come back and Run all again.


In [ ]:
#@title 1) Install node, openclaw, ffmpeg tools (4-6 min)
import json as _json, os, subprocess, urllib.request

def sh(cmd, quiet=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not quiet and r.returncode != 0:
        print("err:", cmd[:60], (r.stderr or r.stdout)[-300:])
    return r

print("ffmpeg:", sh("ffmpeg -version | head -1", quiet=True).stdout.strip()[:50])

if "v24" not in sh("node -v", quiet=True).stdout:
    idx = _json.loads(urllib.request.urlopen("https://nodejs.org/dist/index.json").read())
    ver = [x["version"] for x in idx if x["version"].startswith("v24")][0]
    print("node version:", ver)
    sh("curl -fsSL https://nodejs.org/dist/" + ver + "/node-" + ver + "-linux-x64.tar.xz -o /tmp/node.tar.xz")
    sh("tar -xJf /tmp/node.tar.xz -C /usr/local --strip-components=1")

print("node:", sh("node -v", quiet=True).stdout.strip())
sh("npm i -g openclaw@latest --no-audit --no-fund 2>&1 | tail -3")
print("openclaw:", sh("openclaw --version 2>&1 | head -1", quiet=True).stdout.strip())
sh("pip install -q yt-dlp faster-whisper 2>&1 | tail -2")
print("yt-dlp:", sh("yt-dlp", quiet=True).stdout.strip())

In [ ]:
#@title 2) Bot files + free model router
import json, os, subprocess

BUNDLE = "https://base44.app/api/apps/6afd5d7f94aa00438e512b21/files/mp/public/6afd5d7f94aa00438e512b21/a57c5e21a_openclaw-vm-bundletar.gz"

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(("ok  " if r.returncode == 0 else "err ") + cmd[:70])
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-500:])
    return r

if not os.path.isdir("/tmp/freellmapi"):
    run("git clone --depth 1 https://github.com/tashfeenahmed/freellmapi /tmp/freellmapi")
run("cd /tmp/freellmapi/server && npm install --no-audit --no-fund >/dev/null 2>&1")
run("cd /tmp/freellmapi/server && npm run build >/dev/null 2>&1")

run("curl -fsSL " + BUNDLE + " -o /tmp/b.tgz")
os.makedirs("/tmp/bx", exist_ok=True)
subprocess.run("tar -xzf /tmp/b.tgz -C /tmp/bx", shell=True)
run("mkdir -p ~/.openclaw && cp -rf /tmp/bx/oc/. ~/.openclaw/")
run("mkdir -p /tmp/freellmapi/server/data && cp -f /tmp/bx/freellm/freeapi.db* /tmp/freellmapi/server/data/")

TOKEN = ""
try:
    from google.colab import userdata
    TOKEN = userdata.get("TELEGRAM_BOT_TOKEN") or ""
except Exception:
    TOKEN = ""
if not TOKEN:
    TOKEN = input("Bot token paste karein: ").strip()

p = os.path.expanduser("~/.openclaw/openclaw.json")
cfg = json.loads(open(p).read().replace("/root", os.path.expanduser("~")))
tg = cfg.setdefault("channels", {}).setdefault("telegram", {})
tg["botToken"] = TOKEN
tg["enabled"] = True
json.dump(cfg, open(p, "w"), indent=2)
print("token saved:", bool(TOKEN))

In [ ]:
#@title 1) Install node, openclaw, ffmpeg tools (4-6 min)
import json as _json, os, subprocess, urllib.request

def sh(cmd, quiet=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not quiet and r.returncode != 0:
        print("err:", cmd[:60], (r.stderr or r.stdout)[-300:])
    return r

print("ffmpeg:", sh("ffmpeg -version | head -1", quiet=True).stdout.strip()[:50])

if "v24" not in sh("node -v", quiet=True).stdout:
    idx = _json.loads(urllib.request.urlopen("https://nodejs.org/dist/index.json").read())
    ver = [x["version"] for x in idx if x["version"].startswith("v24")][0]
    print("node version:", ver)
    sh("curl -fsSL https://nodejs.org/dist/" + ver + "/node-" + ver + "-linux-x64.tar.xz -o /tmp/node.tar.xz")
    sh("tar -xJf /tmp/node.tar.xz -C /usr/local --strip-components=1")

print("node:", sh("node -v", quiet=True).stdout.strip())
sh("npm i -g openclaw@latest --no-audit --no-fund 2>&1 | tail -3")
print("openclaw:", sh("openclaw --version 2>&1 | head -1", quiet=True).stdout.strip())
sh("pip install -q yt-dlp faster-whisper 2>&1 | tail -2")
print("yt-dlp:", sh("yt-dlp --version", quiet=True).stdout.strip())